# 🌍 Análise de mudanças climáticas

Notebook para coletar, tratar, analisar e visualizar séries históricas diárias de temperatura e precipitação usando a API **Open-Meteo Archive**.

**Melhorias desta versão:** validação dos parâmetros, requisições com timeout e tentativas automáticas, tratamento explícito de dados ausentes, indicadores de cobertura, tendência em °C por década e exportação organizada.

> **Nota metodológica:** dados de um único ponto de coordenadas representam uma estimativa para a localidade consultada, não necessariamente toda a área municipal. Tendências históricas observadas não demonstram, isoladamente, causalidade.


## 1. Bibliotecas e configurações

In [1]:
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Exibição dos gráficos no Jupyter
import plotly.io as pio
pio.renderers.default = "notebook_connected"


## 2. Cidade e período de análise

In [2]:
CIDADES = {
    "São Paulo (SP)": {"latitude": -23.5505, "longitude": -46.6333},
    "Rio de Janeiro (RJ)": {"latitude": -22.9068, "longitude": -43.1729},
    "Brasília (DF)": {"latitude": -15.7801, "longitude": -47.9292},
    "Manaus (AM)": {"latitude": -3.1190, "longitude": -60.0217},
    "Porto Alegre (RS)": {"latitude": -30.0346, "longitude": -51.2177},
    "Salvador (BA)": {"latitude": -12.9714, "longitude": -38.5014},
    "Fortaleza (CE)": {"latitude": -3.7319, "longitude": -38.5267},
}

# Parâmetros editáveis
cidade_nome = "São Paulo (SP)"
ano_inicio = 1980
ano_fim = 2025  # Use anos completos para comparações anuais
fuso_horario = "America/Sao_Paulo"

if cidade_nome not in CIDADES:
    raise ValueError(f"Cidade não cadastrada: {cidade_nome}")

if not (1900 <= ano_inicio <= ano_fim <= pd.Timestamp.now().year):
    raise ValueError("Informe anos válidos, com ano_inicio <= ano_fim.")

latitude = CIDADES[cidade_nome]["latitude"]
longitude = CIDADES[cidade_nome]["longitude"]
data_inicio = f"{ano_inicio}-01-01"
data_fim = f"{ano_fim}-12-31"

print(f"Localidade: {cidade_nome}")
print(f"Coordenadas: {latitude}, {longitude}")
print(f"Período solicitado: {data_inicio} a {data_fim}")


Localidade: São Paulo (SP)
Coordenadas: -23.5505, -46.6333
Período solicitado: 1980-01-01 a 2025-12-31


## 3. Coleta dos dados na API Open-Meteo

In [3]:
URL_API = "https://archive-api.open-meteo.com/v1/archive"

def criar_sessao():
    """Cria sessão HTTP com tentativas automáticas para erros temporários."""
    tentativas = Retry(
        total=4,
        connect=4,
        read=4,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adaptador = HTTPAdapter(max_retries=tentativas)
    sessao = requests.Session()
    sessao.mount("https://", adaptador)
    return sessao

def carregar_dados_clima(latitude, longitude, data_inicio, data_fim, timezone):
    parametros = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": data_inicio,
        "end_date": data_fim,
        "daily": ["temperature_2m_mean", "precipitation_sum"],
        "timezone": timezone,
    }

    with criar_sessao() as sessao:
        resposta = sessao.get(URL_API, params=parametros, timeout=(10, 90))
        resposta.raise_for_status()
        dados = resposta.json()

    if "daily" not in dados or not dados["daily"].get("time"):
        raise ValueError("A API não retornou dados diários para os parâmetros informados.")

    diario = dados["daily"]
    df = pd.DataFrame({
        "data": pd.to_datetime(diario["time"], errors="coerce"),
        "temperatura_media_c": diario["temperature_2m_mean"],
        "precipitacao_mm": diario["precipitation_sum"],
    })

    df = df.dropna(subset=["data"]).sort_values("data").reset_index(drop=True)
    df["ano"] = df["data"].dt.year
    df["mes"] = df["data"].dt.month

    # Valores ausentes são preservados para permitir medir a cobertura dos dados.
    for coluna in ["temperatura_media_c", "precipitacao_mm"]:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    return df, dados.get("latitude"), dados.get("longitude"), dados.get("timezone")

df, latitude_api, longitude_api, timezone_api = carregar_dados_clima(
    latitude, longitude, data_inicio, data_fim, fuso_horario
)

print(f"Registros diários retornados: {len(df):,}")
print(f"Coordenadas efetivamente utilizadas pela API: {latitude_api}, {longitude_api}")
print(f"Fuso horário informado pela API: {timezone_api}")
display(df.head())


Registros diários retornados: 16,802
Coordenadas efetivamente utilizadas pela API: -23.514938, -46.610504
Fuso horário informado pela API: America/Sao_Paulo


,data,temperatura_media_c,precipitacao_mm,ano,mes
0,1980-01-01,16.90,4.10,1980,1
1,1980-01-02,17.10,0.20,1980,1
2,1980-01-03,18.00,0.00,1980,1
3,1980-01-04,17.80,0.00,1980,1
4,1980-01-05,18.50,0.00,1980,1


## 4. Qualidade e cobertura dos dados

In [4]:
# Quantidade de dias esperados em cada ano dentro do intervalo solicitado
dias_esperados = (
    pd.date_range(data_inicio, data_fim, freq="D")
    .to_series()
    .groupby(lambda data: data.year)
    .size()
    .rename("dias_esperados")
)

cobertura = df.groupby("ano").agg(
    dias_retornados=("data", "nunique"),
    dias_com_temperatura=("temperatura_media_c", "count"),
    dias_com_precipitacao=("precipitacao_mm", "count"),
)

cobertura = cobertura.join(dias_esperados)
cobertura["cobertura_temperatura_pct"] = (
    100 * cobertura["dias_com_temperatura"] / cobertura["dias_esperados"]
)
cobertura["cobertura_precipitacao_pct"] = (
    100 * cobertura["dias_com_precipitacao"] / cobertura["dias_esperados"]
)

display(cobertura.round(1))

print("\nValores ausentes:")
display(df[["temperatura_media_c", "precipitacao_mm"]].isna().sum().rename("quantidade"))


,dias_retornados,dias_com_temperatura,dias_com_precipitacao,dias_esperados,cobertura_temperatura_pct,cobertura_precipitacao_pct
ano,,,,,,
1980,366,366,366,366,100.00,100.00
1981,365,365,365,365,100.00,100.00
1982,365,365,365,365,100.00,100.00
1983,365,365,365,365,100.00,100.00
1984,366,366,366,366,100.00,100.00
1985,365,365,365,365,100.00,100.00
1986,365,365,365,365,100.00,100.00
1987,365,365,365,365,100.00,100.00
1988,366,366,366,366,100.00,100.00



Valores ausentes:


,quantidade
temperatura_media_c,0
precipitacao_mm,0


## 5. Agregação anual e estatísticas

Para evitar comparar totais de precipitação de anos incompletos, o acumulado anual é calculado apenas quando há cobertura diária de precipitação de **100%**. A temperatura média anual é considerada válida com cobertura mínima de **90%**. Esses limites podem ser ajustados conforme o protocolo do estudo.


In [5]:
MIN_COBERTURA_TEMPERATURA = 90.0
MIN_COBERTURA_PRECIPITACAO = 100.0

anual = df.groupby("ano").agg(
    temperatura_media_c=("temperatura_media_c", "mean"),
    precipitacao_total_bruta_mm=("precipitacao_mm", "sum"),
).join(cobertura)

anual["temperatura_media_c"] = anual["temperatura_media_c"].where(
    anual["cobertura_temperatura_pct"] >= MIN_COBERTURA_TEMPERATURA
)
anual["precipitacao_total_mm"] = anual["precipitacao_total_bruta_mm"].where(
    anual["cobertura_precipitacao_pct"] >= MIN_COBERTURA_PRECIPITACAO
)

anual = anual.reset_index()

display(anual[[
    "ano", "temperatura_media_c", "precipitacao_total_mm",
    "cobertura_temperatura_pct", "cobertura_precipitacao_pct"
]].round(2))

temp_media_geral = df["temperatura_media_c"].mean()
precipitacao_media_anual = anual["precipitacao_total_mm"].mean()

print(f"Temperatura média diária no período: {temp_media_geral:.2f} °C")
print(f"Precipitação média dos anos válidos: {precipitacao_media_anual:.1f} mm/ano")
print(f"Anos com precipitação anual válida: {anual['precipitacao_total_mm'].notna().sum()}")


,ano,temperatura_media_c,precipitacao_total_mm,cobertura_temperatura_pct,cobertura_precipitacao_pct
0,1980,19.18,"1,294.80",100.00,100.00
1,1981,18.67,"1,435.50",100.00,100.00
2,1982,18.94,"1,640.30",100.00,100.00
3,1983,19.27,"1,978.20",100.00,100.00
4,1984,19.51,"1,178.10",100.00,100.00
5,1985,19.03,"1,145.10",100.00,100.00
6,1986,19.54,"1,535.30",100.00,100.00
7,1987,19.20,"1,247.90",100.00,100.00
8,1988,18.82,"1,405.50",100.00,100.00
9,1989,18.62,"1,413.00",100.00,100.00


Temperatura média diária no período: 19.37 °C
Precipitação média dos anos válidos: 1318.8 mm/ano
Anos com precipitação anual válida: 46


## 6. Tendência da temperatura

In [6]:
serie_temp = anual.dropna(subset=["temperatura_media_c"]).copy()

if len(serie_temp) >= 2:
    x = serie_temp["ano"].to_numpy(dtype=float)
    y = serie_temp["temperatura_media_c"].to_numpy(dtype=float)

    inclinacao, intercepto = np.polyfit(x, y, 1)
    tendencia = inclinacao * x + intercepto
    r2 = 1 - np.sum((y - tendencia) ** 2) / np.sum((y - y.mean()) ** 2) if len(y) > 2 and not np.isclose(y.var(), 0) else np.nan

    print(f"Tendência linear estimada: {inclinacao * 10:+.3f} °C por década")
    print(f"R² da regressão linear: {r2:.3f}" if pd.notna(r2) else "R² não calculável.")
else:
    raise ValueError("Não há anos válidos suficientes para estimar a tendência.")

fig_temp = px.line(
    serie_temp,
    x="ano",
    y="temperatura_media_c",
    markers=True,
    title=f"Temperatura média anual — {cidade_nome}",
    labels={"ano": "Ano", "temperatura_media_c": "Temperatura média (°C)"},
)
fig_temp.add_trace(go.Scatter(
    x=x, y=tendencia, mode="lines", name="Tendência linear",
    line={"dash": "dash"}
))
fig_temp.update_layout(hovermode="x unified")
fig_temp.show()


Tendência linear estimada: +0.119 °C por década
R² da regressão linear: 0.131


## 7. Precipitação anual

In [7]:
serie_precip = anual.dropna(subset=["precipitacao_total_mm"]).copy()

fig_precip = px.bar(
    serie_precip,
    x="ano",
    y="precipitacao_total_mm",
    title=f"Precipitação acumulada anual — {cidade_nome}",
    labels={"ano": "Ano", "precipitacao_total_mm": "Precipitação acumulada (mm)"},
)
fig_precip.update_layout(xaxis={"dtick": 1}, hovermode="x")
fig_precip.show()


## 8. Sazonalidade mensal

In [8]:
mensal = df.groupby("mes", as_index=False).agg(
    temperatura_media_c=("temperatura_media_c", "mean"),
    precipitacao_media_diaria_mm=("precipitacao_mm", "mean"),
)

fig_mensal = px.bar(
    mensal,
    x="mes",
    y="precipitacao_media_diaria_mm",
    title=f"Precipitação média diária por mês — {cidade_nome}",
    labels={"mes": "Mês", "precipitacao_media_diaria_mm": "Média diária (mm)"},
)
fig_mensal.update_layout(xaxis={"dtick": 1})
fig_mensal.show()


## 9. Exportação dos resultados

In [9]:
# Cria uma pasta de saída e nomes de arquivo seguros
PASTA_SAIDA = Path("resultados_climabrasil")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

slug_cidade = re.sub(r"[^a-z0-9]+", "_", cidade_nome.lower()).strip("_")

arquivo_diario = PASTA_SAIDA / f"dados_diarios_{slug_cidade}_{ano_inicio}_{ano_fim}.csv"
arquivo_anual = PASTA_SAIDA / f"resumo_anual_{slug_cidade}_{ano_inicio}_{ano_fim}.csv"
arquivo_json = PASTA_SAIDA / f"dados_diarios_{slug_cidade}_{ano_inicio}_{ano_fim}.json"

df.to_csv(arquivo_diario, index=False, encoding="utf-8-sig")
anual.to_csv(arquivo_anual, index=False, encoding="utf-8-sig")
df.to_json(arquivo_json, orient="records", date_format="iso", force_ascii=False, indent=2)

print("Arquivos exportados:")
print(f"- {arquivo_diario}")
print(f"- {arquivo_anual}")
print(f"- {arquivo_json}")


Arquivos exportados:
- resultados_climabrasil/dados_diarios_s_o_paulo_sp_1980_2025.csv
- resultados_climabrasil/resumo_anual_s_o_paulo_sp_1980_2025.csv
- resultados_climabrasil/dados_diarios_s_o_paulo_sp_1980_2025.json


## 10. Interpretação e limitações

- A série representa o ponto de coordenadas consultado e pode não refletir a variabilidade de todo o município.
- A API pode retornar dados ausentes; por isso, os indicadores de cobertura acompanham as agregações.
- Totais anuais de precipitação com cobertura inferior a 100% são ocultados para evitar subestimativas por dias faltantes.
- A regressão linear resume a direção média da série, mas não substitui testes estatísticos de tendência, análise de incerteza ou homogeneização climatológica.
- Para conclusões robustas sobre mudanças climáticas, recomenda-se comparar múltiplas localidades, avaliar sazonalidade e extremos e confrontar os resultados com estações meteorológicas oficiais.

**Fonte dos dados:** Open-Meteo Historical Weather API — https://open-meteo.com/en/docs/historical-weather-api
